# Fine-Tuning Representation Models for Classification
## Supervised Classification
- We'll fine-tune a pretrained BERT model to create a task-specific model similar to the one we used in chapter2
- Compared to the embedding model approach, we will fine-tune both the representation model and the classification head as a single architecture.
- To do so, instead of freezing the model, we allow it to be trainable and update its parameters during training. 
- As illustrated in img3, we will use a pretrained BERT model and add a neural network as a classification head, both of which will be fine-tuned for classification.

### Fine-Tuning a Pretrained BERT Model

- We will be using the same dataset we used in Chapter 4 to fine-tune our model, namely the Rotten Tomatoes dataset, which contains 5,331 positive and 5,331 negative movie reviews from Rotten Tomatoes



In [5]:
pip install "datasets>=2.18.0,<3" transformers>=4.38.2 sentence-transformers>=2.5.1 setfit>=1.0.3 accelerate>=0.27.2 seqeval>=1.2.2

Note: you may need to restart the kernel to use updated packages.


In [18]:
# Step 1: Clear accelerator-managed resources
#trainer.accelerator.clear()

# Step 2: Delete Python objects
#del trainer, embedding_model

# Step 3: Force garbage collection
import gc
gc.collect()

# Step 4: Empty CUDA cache
import torch
torch.cuda.empty_cache()

#### Data

In [4]:
from datasets import load_dataset
toamatoes = load_dataset("rotten_tomatoes")
train_data, test_data = toamatoes["train"], toamatoes["test"]


#### Supervised Classification
##### HuggingFace Trainer
- The first step in our classification task is to select the underlying model we want to use. We use "bert-base-cased", which was pretrained on the English Wikipedia as well as a large dataset consisting of unpublished books.
- We define the number of labels that we want to predict beforehand. This is necessary to create the feedforward neural network that is applied on top of our pretrained model:

In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4853.90it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

Next, we will tokenize our data:

In [21]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

Map: 100%|██████████| 1066/1066 [00:00<00:00, 15558.84 examples/s]


- Before creating the Trainer, we will want to prepare a special DataCollator. A DataCollator is a class that helps us build batches of data but also allows us to apply data augmentation.
- During this process of tokenization, and as shown in Chapter 9, we will add padding to the input text to create equally sized representations. We use DataCollatorWithPadding for that.

In [22]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    """Compute F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1= load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

Next, we instantiate our Trainer:
- The TrainingArguments class defines hyperparameters we want to tune, such as the learning rate and how many epochs (rounds) we want to train. 
- The Trainer is used to execute the training process.

In [27]:


from transformers import TrainingArguments, Trainer

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   eval_strategy="epoch",
   report_to="none"
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)



In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.197280,0.489843,0.842204


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.99s/it]


TrainOutput(global_step=534, training_loss=0.20801622769359346, metrics={'train_runtime': 68.7876, 'train_samples_per_second': 124.005, 'train_steps_per_second': 7.763, 'total_flos': 227605451772240.0, 'train_loss': 0.20801622769359346, 'epoch': 1.0})

We get an F1 score of 0.85, which is quite a bit higher than the task-specific model we used in Chapter 4, which resulted in an F1 score of 0.80. It shows that fine-tuning a model yourself can be more advantageous than using a pretrained model. It only costs us a couple of minutes to train.

### Freeze Layers
- To further showcase the importance of training the entire network, the next example will demonstrate how you can use Hugging Face Transformers to freeze certain layers of your network.
- We will freeze the main BERT model and allow only updates to pass through the classification head. This will be a great comparison as we will keep everything the same, except for freezing specific layers.

In [31]:
import gc
gc.collect()

# Step 4: Empty CUDA cache
import torch
torch.cuda.empty_cache()

In [32]:
# Load the model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3315.25it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

In [33]:
for name, param in model.named_parameters():
    print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

Our pretrained BERT model contains a lot of layers that we can potentially freeze. Inspecting these layers gives insight into the structure of the network and what we might want to freeze:

In [37]:
# We can check whether the model was correctly updated
for name, param in model.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")

Parameter: bert.embeddings.word_embeddings.weight ----- True
Parameter: bert.embeddings.position_embeddings.weight ----- True
Parameter: bert.embeddings.token_type_embeddings.weight ----- True
Parameter: bert.embeddings.LayerNorm.weight ----- True
Parameter: bert.embeddings.LayerNorm.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- True
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- True
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- True
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- True
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.bia

We could choose to only freeze certain layers to speed up computing but still allow the main model to learn from the classification task. Generally, we want frozen layers to be followed by trainable layers.

In [38]:
for name, param in model.named_parameters():

     # Trainable classification head
     if name.startswith("classifier"):
        param.requires_grad = True

      # Freeze everything else
     else:
        param.requires_grad = False

In [39]:
# We can check whether the model was correctly updated
for name, param in model.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")



Parameter: bert.embeddings.word_embeddings.weight ----- False
Parameter: bert.embeddings.position_embeddings.weight ----- False
Parameter: bert.embeddings.token_type_embeddings.weight ----- False
Parameter: bert.embeddings.LayerNorm.weight ----- False
Parameter: bert.embeddings.LayerNorm.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: bert.encoder.layer.0.attention.output

In [40]:
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.697492,0.686295,0.602243


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]


TrainOutput(global_step=534, training_loss=0.6965191230345308, metrics={'train_runtime': 28.6079, 'train_samples_per_second': 298.17, 'train_steps_per_second': 18.666, 'total_flos': 227605451772240.0, 'train_loss': 0.6965191230345308, 'epoch': 1.0})

F1 score is 0.60, which is quite a bit lower compared to our original 0.85 score. Instead of freezing nearly all layers, let’s freeze everything up until encoder block 10 as illustrated in Figure 11-6, and see how it affects performance. A major benefit is that this reduces computation but still allows updates to flow through part of the pretrained model:

### Freeze blocks 1-5

In [42]:
# We can check whether the model was correctly updated
for index, (name, param) in enumerate(model.named_parameters()):
     print(f"Parameter: {index}{name} ----- {param.requires_grad}")

Parameter: 0bert.embeddings.word_embeddings.weight ----- False
Parameter: 1bert.embeddings.position_embeddings.weight ----- False
Parameter: 2bert.embeddings.token_type_embeddings.weight ----- False
Parameter: 3bert.embeddings.LayerNorm.weight ----- False
Parameter: 4bert.embeddings.LayerNorm.bias ----- False
Parameter: 5bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: 6bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: 7bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: 8bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: 9bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: 10bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: 11bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: 12bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: 13bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: 14bert.encoder.laye

In [44]:
# Load model
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Encoder block 10 starts at index 165 and
# we freeze everything before that block
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5687.52it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.460971,0.410286,0.818702


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.77s/it]


TrainOutput(global_step=534, training_loss=0.4568156981736087, metrics={'train_runtime': 33.3181, 'train_samples_per_second': 256.017, 'train_steps_per_second': 16.027, 'total_flos': 227605451772240.0, 'train_loss': 0.4568156981736087, 'epoch': 1.0})

We got an F1 score of 0.81, which is much higher than our previous score of 0.61 when freezing all layers. It demonstrates that although we generally want to train as many layers as possible, you can get away with training less if you do not have the necessary computing power.

In [ ]:
scores = []
for index in range(12):
    # Re-load model
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

    # Freeze encoder blocks 0-index
    for name, param in model.named_parameters():
        if "layer" in name:
            layer_nr = int(name.split("layer")[1].split(".")[1])
            if layer_nr <= index:
                param.requires_grad = False
            else:
                param.requires_grad = True

#       # Train
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_test,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            )
        trainer.train()
        # Evaluate to get metrics including F1
        eval_results = trainer.evaluate()
        # Extract F1 score
        f1_score = eval_results['eval_f1']
        scores.append(f1_score)


        

## Few-Shot Classification
- Few-Shot classification is a technique within superbvised classification where you have a classifier learn target labels based on only few labeled examples.
- This technique is great when you have a classification task but do not have many labeled data points readily available.
- In other words, this method allows you to label a few high-quality data points per class on which to train the model. 
- This idea of using a few labeled data points for training your model is shown in img4.
### SetFit: Efficient Fine-Tuning with Few Training Examples
- To perform few-shot text classification, we use an efficient framework called SetFit.2 It is built on top of the architecture of sentence-transformers to generate high-quality textual representations that are updated during training: https://github.com/huggingface/setfit
- Only a few labeled examples are needed for this framework to be competitive with fine-tuning a BERT-like model on a large, labeled dataset as we explored in the previous example.
- The underlying algorithm of SetFit consists of three steps:
  - Sampling training data: Based on in-class and out-class selection of labeled data it generates positive (similar) and negative (dissimilar) pairs of sentences
  - Fine-tuning embeddings: Fine-tuning a pretrained embedding model based on the previously generated training data
  - Training a classifier: Create a classification head on top of the embedding model and train it using the previously generated training data 

- Say, for example, we have the training dataset in img5 that classifies text into two categories: text about programming languages, and text about pets.
- In step 1, SetFit handles this problem by generating the necessary data based on in-class and out-class selection as we illustrate in Figure 6. For example, when we have 16 sentences about sports, we can create 16 * (16 – 1) / 2 = 120 pairs that we label as positive pairs. We can use this process to generate negative pairs by collecting pairs from different classes.

- In step 2, we can use the generated sentence pairs to fine-tune the embedding model. This leverages a method called **contrastive learning** to fine-tune a pretrained BERT model. As we reviewed in Chapter 10, contrastive learning allows accurate sentence embeddings to be learned from pairs of similar (positive) and dissimilar (negative) sentences.

Since we generated these pairs in the previous step, we can use them to fine-tune a SentenceTransformers model. Although we have discussed contrastive learning before, we again illustrate the method in img 7 as a refresher.

The goal of fine-tuning this embedding model is that it can create embeddings that are tuned to the classification task. The relevance of the classes, and their relative meaning, are distilled into the embeddings through fine-tuning the embedding model.

- In step 3, we generate embeddings for all sentences and use those as the input of a classifier. We can use the fine-tuned SentenceTransformers model to convert our sentences into embeddings that we can use as features. The classifier learns from our fine-tuned embeddings to accurately predict unseen sentences. This last step is illustrated in 

### Fine-Tuning for Few-Shot Classification




In [4]:
pip install setfit>=1.0.4

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from setfit import sample_dataset

# We simulate a few-shot setting by sampling 16 examples per class
sampled_train_data = sample_dataset(tomatoes["train"], num_samples=16)

In [ ]:
from setfit import SetFitModel

# Load a pre-trained SentenceTransformer model
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")



- After loading in the pretrained SentenceTransformer model, we can start defining our SetFitTrainer. By default, a logistic regression model is chosen as the classifier to train.
- Similar to what we did with Hugging Face Transformers, we can use the trainer to define and play around with relevant parameters. For example, we set the num_epochs to 3 so that contrastive learning will be performed for three epochs

In [ ]:
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

# Define training arguments
args = SetFitTrainingArguments(
    num_epoch=3, # The number of epochs to use for contrastive learning
    num_iterations=20, # the number of texts pairs to generate
)

args.eval_startegy = args.evaluation_strategy

# Create trainer
trainer = SetFitTrainer(
    model=model,
    args=args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric = "f1"
)

In [ ]:
trainer.train()

***** Running training *****
- Num unique pairs = 1280
- Batch size = 16
- Num epochs = 3
- Total optimization steps = 240

Notice that the output mentions that 1,280 sentence pairs were generated for fine-tuning the SentenceTransformer model. As a default, 20 sentence pair combinations are generated for each sample in our data, which would be 20 * 32 = 680 samples. We will have to multiply this value by 2 for each positive and negative pair generated, 680 * 2 = 1,280 sentence pairs. Generating 1,280 sentence pairs is quite impressive considering we only had 32 labeled sentences to start with!

### tip
When we do not specifically define a classification head, by default a logistic regression is used. If we would like to specify a classification head ourselves, we can do so by specifying the following model in SetFitTrainer:
```bash
# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-mpnet-base-v2",
    use_differentiable_head=True,
    head_params={"out_features": num_classes},
)

# Create trainer
trainer = SetFitTrainer(
    model=model,
    ...
)
```

In [ ]:
# Evaluate the model on our test data
trainer.evaluate()

```batch
{'f1': 0.8363988383349468}
```
With only 32 labeled documents, we get an F1 score of 0.85. Considering that the model was trained on a tiny subset of the original data, this is very impressive! Moreover, in Chapter 2, we got the same performance but instead trained a logistic regression model on the embeddings of the full data. Thus, this pipeline demonstrates the potential of taking the time to label just a few instances.

 ## Continued Pretraining with Masked Language Modeling (MLM)
 


In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Load model for Masked Language Modeling (MLM)
model = AutoModelForMaskedLM.from_pretrained("bert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3740.75it/s]
BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
from datasets import load_dataset
toamatoes = load_dataset("rotten_tomatoes")
train_data, test_data = toamatoes["train"], toamatoes["test"]

In [6]:
def preprocess_function(examples):
   return tokenizer(examples["text"], truncation=True)

# Tokenize data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_train = tokenized_train.remove_columns("label")
tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_test = tokenized_test.remove_columns("label")

Map: 100%|██████████| 1066/1066 [00:00<00:00, 23688.97 examples/s]


Previously, we used DataCollatorWithPadding, which dynamically pads the input it receives.

Instead, we will have a DataCollator that will perform the masking of tokens for us. 
- There are two methods that are generally used for this: **token masking**  and **whole-word masking**. 
    - With token masking, we randomly mask 15% of the tokens in a sentence. It might happen that part of a word will be masked. 
    - To enable masking of the entire word, we could apply whole-word masking, as illustrated in img11


- Generally, predicting whole words tends to be more complicated than tokens, which makes the model perform better as it needs to learn more accurate and precise representations during training. 
- However, it tends to take a bit more time to converge. We will be going with token masking in this example using **DataCollatorForLanguageModeling** for faster convergence. 
- However, we can use whole-word masking by replacing **DataCollatorForLanguageModeling** with **DataCollatorForWholeWordMask**. Lastly, we set the probability that a token is masked in a given sentence to 15% (mlm_probability):



In [7]:
from transformers import DataCollatorForLanguageModeling

# Masking tokens
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [10]:
from transformers import TrainingArguments, Trainer
# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=10,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

In [11]:
# Save pre-trained tokenizer
tokenizer.save_pretrained("mlm")

# Train model
trainer.train()

# Save updated model
model.save_pretrained("mlm")

Step,Training Loss
500,2.594836
1000,2.380089
1500,2.302740
2000,2.191057
2500,2.148519
3000,2.096211
3500,2.061113
4000,1.987618
4500,1.980846
5000,1.960658


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.72s/it]


This gives us an updated model in the mlm folder. To evaluate its performance we would normally fine-tune the model on a variety of tasks. For our purposes, however, we can run some masking tasks to see if it has learned from its continued training.

- We will do so by loading in our pretrained model before we continue pretraining. Using the sentence "What a horrible [MASK]!" the model will predict which word would be in place of "[MASK]"

In [12]:
from transformers import pipeline

# Load and create predictions
mask_filler = pipeline("fill-mask", model="bert-base-cased")
preds = mask_filler("What a horrible [MASK]!")

# Print results
for pred in preds:
    print(f">>> {pred['sequence']}")



Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3107.66it/s]
BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


>>> What a horrible idea!
>>> What a horrible dream!
>>> What a horrible thing!
>>> What a horrible day!
>>> What a horrible thought!


The output demonstrates concepts like “idea,” “dream,” and “day,” which definitely make sense. Next, let’s see what our updated model predicts:

In [13]:
# Load and create predictions
mask_filler = pipeline("fill-mask", model="mlm")
preds = mask_filler("What a horrible [MASK]!")

# Print results
for pred in preds:
    print(f">>> {pred['sequence']}")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 563.24it/s]


>>> What a horrible movie!
>>> What a horrible film!
>>> What a horrible mess!
>>> What a horrible story!
>>> What a horrible comedy!


The next step would be to fine-tune this model on the classification task that we did at the beginning of this chapter. Simply load the model as follows and you are good to go:

In [14]:
from transformers import AutoModelForSequenceClassification

# Fine-tune for classification
model = AutoModelForSequenceClassification.from_pretrained("mlm", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("mlm")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4191.30it/s]
BertForSequenceClassification LOAD REPORT from: mlm
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your

In [17]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)




Map: 100%|██████████| 1066/1066 [00:00<00:00, 20101.46 examples/s]


In [18]:
import numpy as np
import evaluate


def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

In [23]:
from transformers import TrainingArguments, Trainer

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   eval_strategy="epoch",
   report_to="none"
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.176273,0.584568,0.856089


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.45s/it]


TrainOutput(global_step=534, training_loss=0.19144025366851006, metrics={'train_runtime': 63.7388, 'train_samples_per_second': 133.827, 'train_steps_per_second': 8.378, 'total_flos': 227605451772240.0, 'train_loss': 0.19144025366851006, 'epoch': 1.0})

# Get Predictions using MLM model created

In [25]:
import torch

def predict_review(text, model, tokenizer):
    """
    Predict the sentiment of a movie review
    
    Args:
        text: Movie review text
        model: Trained model
        tokenizer: Tokenizer
    
    Returns:
        dict with prediction and confidence
    """
    # Tokenize the input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # Move to same device as model
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get prediction
    model.eval()  # Set to evaluation mode
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predicted_class = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0][predicted_class].item()
    
    # Map to label
    label_map = {0: "negative", 1: "positive"}
    
    return {
        "text": text,
        "label": label_map[predicted_class],
        "confidence": confidence,
        "probabilities": {
            "negative": probabilities[0][0].item(),
            "positive": probabilities[0][1].item()
        }
    }

# Test with example reviews
reviews = [
    "This movie was absolutely amazing! Best film I've seen this year.",
    "Terrible movie, waste of time and money.",
    "It was okay, nothing special but not bad either."
]

for review in reviews:
    result = predict_review(review, model, tokenizer)
    print(f"\nReview: {result['text'][:60]}...")
    print(f"Prediction: {result['label']} (confidence: {result['confidence']:.2%})")
    print(f"Probabilities: Negative={result['probabilities']['negative']:.2%}, Positive={result['probabilities']['positive']:.2%}")


Review: This movie was absolutely amazing! Best film I've seen this ...
Prediction: positive (confidence: 99.78%)
Probabilities: Negative=0.22%, Positive=99.78%

Review: Terrible movie, waste of time and money....
Prediction: negative (confidence: 99.84%)
Probabilities: Negative=99.84%, Positive=0.16%

Review: It was okay, nothing special but not bad either....
Prediction: negative (confidence: 85.62%)
Probabilities: Negative=85.62%, Positive=14.38%


## Named-Entity Recogntion
- In this section, we will delve into the process of fine-tuning a pretrained BERT model specifically for NER (named-entity recognition).
- Instead of classifying entire documents, this procedure allows for the classification of individual tokens and/or words, including people and locations. 
- This is especially helpful for de-identification and anonymization tasks when there is sensitive data. Check img12.
- Fine-tuning the pretrained BERT model follows a similar architecture akin to what we observed with document classification. However, there is a fundamental shift in the classification approach. 
- Rather than relying on the aggregation or pooling of token embeddings, the model now makes predictions for individual tokens in a sequence. It is crucial to emphasize that our word-level classification task does not entail classifying entire words, but rather the tokens that collectively constitute those words. img 13 provides a visual representation of this token-level classification.

- Here are a number of intersting datasets we can explore for NER:
   - tner/mit_movie_trivia
   - tner/mit_restaurant
   - wnut_17
   - conll2003
### Preparing data for Named-Entity Recognition

- We'll use the english veersion of the CoNLL-2003 dataset, which contains several different types of named entities (person, organization, location, miscellaneous, and no entity) and has roughly 14,000 training samples
- First thing: Clean the VRAM


In [1]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [4]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import numpy as np

In [6]:
from datasets import load_dataset
# The CoNLL-2003 dataset for NER
dataset = load_dataset("conll2003", trust_remote_code=True)

Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 4895.99 examples/s]


In [7]:
example = dataset["train"][848]
example

{'id': '848',
 'tokens': ['Dean',
  'Palmer',
  'hit',
  'his',
  '30th',
  'homer',
  'for',
  'the',
  'Rangers',
  '.'],
 'pos_tags': [22, 22, 38, 29, 16, 21, 15, 12, 23, 7],
 'chunk_tags': [11, 12, 21, 11, 12, 12, 13, 11, 12, 0],
 'ner_tags': [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]}

- This dataset provides us with labels for each word given in a sentence. These labels can be found in the ner_tags key, which refers to the following possible entities:

In [8]:
label2id = {
    "O":0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5, 
    "I-LOC": 6, 
    "B-MISC": 7, 
    "I-MISC": 8
}

id2label = {index:label for label, index in label2id.items()}
id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC',
 7: 'B-MISC',
 8: 'I-MISC'}

In [9]:
label2id

{'O': 0,
 'B-PER': 1,
 'I-PER': 2,
 'B-ORG': 3,
 'I-ORG': 4,
 'B-LOC': 5,
 'I-LOC': 6,
 'B-MISC': 7,
 'I-MISC': 8}

- These entities correspond to specific categories: a person (PER), organization (ORG), location (LOC), miscellaneous entities (MISC), and no entity (O). 
- Note that these entities are prefixed with either a B (beginning) or an I (inside). If two tokens that follow each other are part of the same phrase, then the start of that phrase is indicated with B, which is followed by an I to show that they belong to each other and are not independent entities.
- This process is further illustrated in Figure 14. In the figure, since “Dean” is the start of the phrase and “Palmer” is the end, we know that “Dean Palmer” is a person and that “Dean” and “Palmer” are not individual people.

In [10]:
# Load tokdenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Load the model
# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1641.65it/s]
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expec

- Let’s explore how the tokenizer would process our example:

In [12]:
token_ids = tokenizer(example['tokens'], is_split_into_words=True)['input_ids']
token_ids

[101, 4285, 8450, 1855, 1117, 13631, 1313, 1197, 1111, 1103, 6838, 119, 102]

In [13]:
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)
sub_tokens

['[CLS]',
 'Dean',
 'Palmer',
 'hit',
 'his',
 '30th',
 'home',
 '##r',
 'for',
 'the',
 'Rangers',
 '.',
 '[SEP]']

- The tokenizer added the [CLS] and [SEP] tokens as we learned in Chapters 2 and 3. 
- Note that the word 'homer' was further split up into the tokens 'home' and '##r'. 
- This creates a bit of a problem for us since we have labeled data at **the word level but not at the token level**. This can be resolved by aligning the labels with their subtoken counterparts during tokenization.
- Let’s consider the word 'Maarten', which has the label B-PER to signal that this is a person. 
- If we pass that word through the tokenizer, it splits the word up into the tokens 'Ma', '##arte', and '##n'. 
  - We cannot use the B-PER entity for all tokens as that would signal that the three tokens are all independent people. 
  - Whenever an entity is split into tokens, the first token should have B (for beginning) and the following should be I (for inner).

- Therefore, 'Ma' will get the B-PER to signal the start of a phrase, and '##arte', and '##n' will get the I-PER to signal they belong to a phrase. This alignment process is illustrated in img15.

In [14]:
def align_labels(examples):
    token_ids = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = examples["ner_tags"]

    updated_labels = []
    for index, label in enumerate(labels):

        # Map tokens to their respective word
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:

            # The start of a new word
            if word_idx != previous_word_idx:

                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)

            # Special token is -100
            elif word_idx is None:
                label_ids.append(-100)

            # If the label is B-XXX we change it to I-XXX
            else:
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)

        updated_labels.append(label_ids)

    token_ids["labels"] = updated_labels
    return token_ids




In [15]:
tokenized = dataset.map(align_labels, batched=True)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 10524.52 examples/s]


In [16]:
# Difference between original and updated labels
print(f"Original: {example['ner_tags']}")
print(f"Updated: {tokenized['train'][848]['labels']}")

Original: [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]
Updated: [-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]


- Now that we have tokenized and aligned the labels, we can start thinking about defining our evaluation metrics. 
- This is also different from what we have seen before. Instead of a single prediction per document, we now have multiple predictions per document, namely per token.

In [17]:
import evaluate

# Load sequential evaluation
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    # Create predictions
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    # Document-level iteration
    for prediction, label in zip(predictions, labels):

      # token-level iteration
      for token_prediction, token_label in zip(prediction, label):

        # We ignore special tokens
        if token_label != -100:
          true_predictions.append([id2label[token_prediction]])
          true_labels.append([id2label[token_label]])

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {"f1": results["overall_f1"]}



### Fine-Tuning for Named-Entity Recognition
We are nearly there. Instead of DataCollatorWithPadding, we need a collator that works with classification on a token level, namely DataCollatorForTokenClassification:


In [18]:
from transformers import DataCollatorForTokenClassification

# Token-classification Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [28]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   eval_strategy="epoch",
   report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# 2. Train (run this cell)
trainer.train()



Epoch,Training Loss,Validation Loss,F1
1,0.017218,0.196469,0.913113


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.63s/it]


TrainOutput(global_step=878, training_loss=0.02254981277743885, metrics={'train_runtime': 107.693, 'train_samples_per_second': 130.38, 'train_steps_per_second': 8.153, 'total_flos': 351240792638148.0, 'train_loss': 0.02254981277743885, 'epoch': 1.0})

In [23]:
trainer.train()

Step,Training Loss
500,0.109376


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.79s/it]


TrainOutput(global_step=878, training_loss=0.09386709339254809, metrics={'train_runtime': 103.8886, 'train_samples_per_second': 135.154, 'train_steps_per_second': 8.451, 'total_flos': 351240792638148.0, 'train_loss': 0.09386709339254809, 'epoch': 1.0})

In [29]:
# Save our fine-tuned model
trainer.save_model("ner_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.54s/it]


In [36]:
from transformers import pipeline
# Run inference on the fine-tuned model
token_classifier = pipeline(
    "token-classification",
    model="ner_model",
)
results= token_classifier("Bilal Ben Mahria spent 3 years at Upwork. I am java developer located in Morocco")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4522.69it/s]


In [37]:
results

[{'entity': 'B-PER',
  'score': 0.9991099,
  'index': 1,
  'word': 'B',
  'start': 0,
  'end': 1},
 {'entity': 'I-PER',
  'score': 0.9995271,
  'index': 2,
  'word': '##ila',
  'start': 1,
  'end': 4},
 {'entity': 'I-PER',
  'score': 0.9996362,
  'index': 3,
  'word': '##l',
  'start': 4,
  'end': 5},
 {'entity': 'I-PER',
  'score': 0.99948126,
  'index': 4,
  'word': 'Ben',
  'start': 6,
  'end': 9},
 {'entity': 'I-PER',
  'score': 0.99949014,
  'index': 5,
  'word': 'Ma',
  'start': 10,
  'end': 12},
 {'entity': 'I-PER',
  'score': 0.9995389,
  'index': 6,
  'word': '##hr',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': 0.99960655,
  'index': 7,
  'word': '##ia',
  'start': 14,
  'end': 16},
 {'entity': 'B-ORG',
  'score': 0.93675244,
  'index': 12,
  'word': 'Up',
  'start': 34,
  'end': 36},
 {'entity': 'I-ORG',
  'score': 0.8964201,
  'index': 13,
  'word': '##work',
  'start': 36,
  'end': 40},
 {'entity': 'B-LOC',
  'score': 0.99937207,
  'index': 22,
  'word': 'Mo

'B-PER'